# 3.6 Lab: FlashAttention Tiling Visualization[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/03_attention_variants/03.6_flash_attention_algorithm/lab.ipynb) [![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.cloud/github/harshuljain13/llm-inference-at-scale/blob/master/content/03_attention_variants/03.6_flash_attention_algorithm/lab.ipynb)This lab visualizes how FlashAttention tiles the attention computation to avoid materializing the full N x N score matrix in HBM. We implement the online softmax algorithm step-by-step and compare HBM traffic between standard and tiled attention.

In [ ]:
import numpy as npimport matplotlib.pyplot as pltimport matplotlib.patches as mpatches# --- Parameters (change these and re-run) ---N = 16          # Sequence length (small for visualization)d = 4           # Head dimensionB_r = 4         # Query block sizeB_c = 4         # Key/Value block sizenp.random.seed(42)

## 1. Standard Attention vs Tiled Access PatternStandard attention materializes the full N x N matrix. FlashAttention processes it in B_r x B_c tiles, never holding more than one tile in SRAM at a time.

In [ ]:
def visualize_tiling_pattern():    """Show which tiles of the NxN matrix are processed and in what order."""    fig, axes = plt.subplots(1, 2, figsize=(12, 5))    # Left: standard attention (full matrix in HBM)    ax = axes[0]    ax.imshow(np.ones((N, N)), cmap='Reds', alpha=0.3, extent=[0, N, N, 0])    ax.set_title('Standard: Full N x N in HBM', fontsize=12)    ax.set_xlabel('Key position')    ax.set_ylabel('Query position')    ax.set_xticks(range(0, N+1, B_c))    ax.set_yticks(range(0, N+1, B_r))    ax.grid(True, alpha=0.3)    # Right: tiled access pattern with processing order    ax = axes[1]    order = 0    n_br = N // B_r  # number of row blocks    n_bc = N // B_c  # number of col blocks    for i in range(n_br):        for j in range(n_bc):            # Color intensity shows processing order            color_val = order / (n_br * n_bc)            rect = mpatches.FancyBboxPatch(                (j * B_c + 0.1, i * B_r + 0.1), B_c - 0.2, B_r - 0.2,                boxstyle="round,pad=0.05",                facecolor=plt.cm.Blues(0.3 + 0.6 * color_val),                edgecolor='black', linewidth=1            )            ax.add_patch(rect)            # Label with processing order            ax.text(j * B_c + B_c/2, i * B_r + B_r/2, str(order),                    ha='center', va='center', fontsize=9, fontweight='bold')            order += 1    ax.set_xlim(0, N)    ax.set_ylim(N, 0)    ax.set_title(f'FlashAttention: {B_r}x{B_c} tiles (one in SRAM at a time)', fontsize=12)    ax.set_xlabel('Key position')    ax.set_ylabel('Query position')    ax.set_xticks(range(0, N+1, B_c))    ax.set_yticks(range(0, N+1, B_r))    ax.grid(True, alpha=0.3)    ax.set_aspect('equal')    plt.tight_layout()    plt.savefig('tiling_pattern.png', dpi=150, bbox_inches='tight')    plt.show()visualize_tiling_pattern()

## 2. Online Softmax: Step-by-StepThe key insight: softmax can be computed incrementally. We maintain running max (m) and sum (l) statistics, correcting previous results when a new maximum is discovered.

In [ ]:
def online_softmax_demo():    """Demonstrate online softmax computing identical results to standard softmax."""    # Generate a single query row's scores against all keys    scores = np.random.randn(N)  # one row of S = q @ K^T    # --- Standard softmax (needs full row) ---    m_global = np.max(scores)    exp_scores = np.exp(scores - m_global)    standard_result = exp_scores / np.sum(exp_scores)    # --- Online softmax (processes in chunks of B_c) ---    m = -np.inf  # running max    l = 0.0      # running sum of exponentials    # Store per-chunk exponentials for visualization    chunk_results = []    for j in range(0, N, B_c):        chunk = scores[j:j+B_c]        m_new = max(m, np.max(chunk))       # update max        # Rescale previous sum with correction factor        l = np.exp(m - m_new) * l + np.sum(np.exp(chunk - m_new))        m = m_new        chunk_results.append({'m': m, 'l': l, 'correction': np.exp(m - m_new) if m != -np.inf else 1.0})    # Final online softmax result    online_result = np.exp(scores - m) / l    # Verify numerical equivalence    max_diff = np.max(np.abs(standard_result - online_result))    print(f"Max difference between standard and online softmax: {max_diff:.2e}")    print(f"Numerically identical: {max_diff < 1e-10}")    # Visualize the running statistics    fig, axes = plt.subplots(1, 3, figsize=(14, 4))    chunks = range(1, len(chunk_results) + 1)    axes[0].bar(chunks, [c['m'] for c in chunk_results], color='#dbeafe', edgecolor='black')    axes[0].set_title('Running Max (m) per Tile')    axes[0].set_xlabel('Tile index')    axes[0].set_ylabel('Max value')    axes[1].bar(chunks, [c['l'] for c in chunk_results], color='#dcfce7', edgecolor='black')    axes[1].set_title('Running Sum (l) per Tile')    axes[1].set_xlabel('Tile index')    # Show final probabilities match    x = np.arange(N)    axes[2].bar(x - 0.2, standard_result, 0.4, label='Standard', color='#f3e8ff', edgecolor='black')    axes[2].bar(x + 0.2, online_result, 0.4, label='Online', color='#fef3c7', edgecolor='black')    axes[2].set_title('Softmax Output (identical)')    axes[2].set_xlabel('Position')    axes[2].legend()    plt.tight_layout()    plt.savefig('online_softmax.png', dpi=150, bbox_inches='tight')    plt.show()online_softmax_demo()

## 3. Full FlashAttention Forward Pass (Tiled)We implement the complete tiled attention algorithm with online softmax, then verify it matches standard attention output exactly.

In [ ]:
def flash_attention_forward(Q, K, V, B_r, B_c):    """    FlashAttention forward pass with online softmax.    Returns output O (N x d), identical to standard attention.    Tracks HBM reads/writes for comparison.    """    N_q, d_dim = Q.shape    O = np.zeros((N_q, d_dim))          # output accumulator    hbm_reads = 0                        # count element reads from HBM    hbm_writes = 0                       # count element writes to HBM    for i in range(0, N_q, B_r):        # Load Q block from HBM to SRAM        Q_i = Q[i:i+B_r]                # B_r x d        hbm_reads += Q_i.size        # Initialize running statistics for this Q block        m_i = np.full(B_r, -np.inf)     # row maxima        l_i = np.zeros(B_r)             # row sums        o_i = np.zeros((B_r, d_dim))    # output accumulator        for j in range(0, N_q, B_c):            # Load K, V blocks from HBM to SRAM            K_j = K[j:j+B_c]            # B_c x d            V_j = V[j:j+B_c]            # B_c x d            hbm_reads += K_j.size + V_j.size            # Compute score tile in SRAM (never written to HBM)            S_ij = Q_i @ K_j.T          # B_r x B_c            # Online softmax update            m_new = np.maximum(m_i, np.max(S_ij, axis=1))            # Correction factor for previous accumulations            correction = np.exp(m_i - m_new)            # Exponentiated scores with new max            P_ij = np.exp(S_ij - m_new[:, None])            # Update running sum            l_i = correction * l_i + np.sum(P_ij, axis=1)            # Rescale previous output and add new contribution            o_i = correction[:, None] * o_i + P_ij @ V_j            # Update max            m_i = m_new        # Final normalization and write to HBM        O[i:i+B_r] = o_i / l_i[:, None]        hbm_writes += B_r * d_dim    return O, hbm_reads, hbm_writesdef standard_attention(Q, K, V):    """Standard attention: materializes full NxN matrix in HBM."""    N_q, d_dim = Q.shape    # HBM traffic: read Q,K,V + write S + read S + write P + read P + write O    S = Q @ K.T                          # N x N written to HBM    hbm_reads = Q.size + K.size + V.size + S.size + S.size    m = np.max(S, axis=1, keepdims=True)    P = np.exp(S - m)    P = P / np.sum(P, axis=1, keepdims=True)    O = P @ V    hbm_writes = S.size + P.size + O.size    return O, hbm_reads, hbm_writes# --- Run both and compare ---Q = np.random.randn(N, d).astype(np.float32)K = np.random.randn(N, d).astype(np.float32)V = np.random.randn(N, d).astype(np.float32)O_flash, flash_reads, flash_writes = flash_attention_forward(Q, K, V, B_r, B_c)O_std, std_reads, std_writes = standard_attention(Q, K, V)print(f"Max output difference: {np.max(np.abs(O_flash - O_std)):.2e}")print(f"\nHBM Traffic Comparison (elements):")print(f"  Standard:       reads={std_reads:,}, writes={std_writes:,}, total={std_reads+std_writes:,}")print(f"  FlashAttention: reads={flash_reads:,}, writes={flash_writes:,}, total={flash_reads+flash_writes:,}")print(f"  Reduction:      {(std_reads+std_writes)/(flash_reads+flash_writes):.2f}x")

## 4. HBM Traffic Scaling: Standard vs FlashAttentionAs sequence length grows, the gap between standard and tiled attention widens because standard attention's N^2 intermediate matrix dominates.

In [ ]:
def hbm_scaling_comparison():    """Compare HBM traffic as N grows for standard vs FlashAttention."""    seq_lengths = [16, 32, 64, 128, 256, 512, 1024, 2048, 4096]    d_fixed = 128    B_r_fixed = 64    B_c_fixed = 64    std_traffic = []    flash_traffic = []    for n in seq_lengths:        # Standard: read Q,K,V (3*N*d) + write S (N^2) + read S + write P + read P + write O        std_total = 3*n*d_fixed + 3*n*n + n*d_fixed  # simplified        std_traffic.append(std_total)        # Flash: Q read once (N*d) + K,V read ceil(N/B_r) times each + O write (N*d)        n_br = (n + B_r_fixed - 1) // B_r_fixed        flash_total = n*d_fixed + 2*n*d_fixed*n_br + n*d_fixed        flash_traffic.append(flash_total)    fig, axes = plt.subplots(1, 2, figsize=(12, 5))    # Absolute traffic    axes[0].plot(seq_lengths, [t/1e6 for t in std_traffic], 'o-',                 color='#991b1b', linewidth=2, label='Standard')    axes[0].plot(seq_lengths, [t/1e6 for t in flash_traffic], 's-',                 color='#166534', linewidth=2, label='FlashAttention')    axes[0].set_xlabel('Sequence Length (N)')    axes[0].set_ylabel('HBM Traffic (M elements)')    axes[0].set_title('HBM Traffic vs Sequence Length')    axes[0].set_xscale('log', base=2)    axes[0].set_yscale('log')    axes[0].legend()    axes[0].grid(True, alpha=0.3)    # Reduction factor    reduction = [s/f for s, f in zip(std_traffic, flash_traffic)]    axes[1].bar(range(len(seq_lengths)), reduction, color='#dbeafe', edgecolor='black')    axes[1].set_xticks(range(len(seq_lengths)))    axes[1].set_xticklabels([str(n) for n in seq_lengths], rotation=45)    axes[1].set_xlabel('Sequence Length (N)')    axes[1].set_ylabel('Reduction Factor (x)')    axes[1].set_title('FlashAttention HBM Reduction Factor')    axes[1].axhline(y=1, color='gray', linestyle='--', alpha=0.5)    axes[1].grid(True, alpha=0.3, axis='y')    plt.tight_layout()    plt.savefig('hbm_scaling.png', dpi=150, bbox_inches='tight')    plt.show()hbm_scaling_comparison()

## 5. Visualizing the Rescaling CorrectionWhen a new tile contains a value larger than the current running max, all previous exponentials must be corrected downward. This visualization shows how the correction factor evolves.

In [ ]:
def visualize_rescaling():    """Show how online softmax correction factors evolve across tiles."""    # Use larger N to see interesting correction patterns    N_demo = 32    scores = np.random.randn(N_demo) * 2  # scale up for visible corrections    B_demo = 8    n_tiles = N_demo // B_demo    # Track statistics per tile    maxes = []    corrections = []    m = -np.inf    for j in range(0, N_demo, B_demo):        chunk = scores[j:j+B_demo]        m_new = max(m, np.max(chunk))        corr = np.exp(m - m_new) if m != -np.inf else 1.0        corrections.append(corr)        maxes.append(m_new)        m = m_new    fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)    tiles = range(n_tiles)    # Running max    axes[0].step(tiles, maxes, where='mid', color='#2563eb', linewidth=2)    axes[0].scatter(tiles, maxes, color='#2563eb', zorder=5, s=60)    axes[0].set_ylabel('Running Max (m)')    axes[0].set_title('Online Softmax: Running Statistics Across Tiles')    axes[0].grid(True, alpha=0.3)    # Highlight tiles where max increased    for i in range(1, n_tiles):        if maxes[i] > maxes[i-1]:            axes[0].axvspan(i-0.4, i+0.4, alpha=0.15, color='red')    # Correction factors    colors = ['#166534' if c == 1.0 else '#991b1b' for c in corrections]    axes[1].bar(tiles, corrections, color=colors, edgecolor='black', alpha=0.8)    axes[1].axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)    axes[1].set_xlabel('Tile Index')    axes[1].set_ylabel('Correction Factor exp(m_old - m_new)')    axes[1].set_ylim(0, 1.1)    axes[1].grid(True, alpha=0.3, axis='y')    # Legend    green_patch = mpatches.Patch(color='#166534', label='No correction (max unchanged)')    red_patch = mpatches.Patch(color='#991b1b', label='Rescale needed (new max found)')    axes[1].legend(handles=[green_patch, red_patch])    plt.tight_layout()    plt.savefig('rescaling.png', dpi=150, bbox_inches='tight')    plt.show()visualize_rescaling()

## Key Takeaways1. FlashAttention tiles the N x N attention computation into SRAM-sized blocks, eliminating HBM writes of the full score matrix.2. Online softmax maintains running max and sum statistics, correcting previous results when a new maximum appears.3. The output is mathematically identical to standard attention: no approximation.4. HBM traffic reduction grows with sequence length, transforming attention from memory-bound to compute-bound.5. FlashAttention-2/3 add parallelism, warp partitioning, and hardware-specific optimizations on top of this core algorithm.